In [ ]:
import numpy as np

LABELS = {"normal": 0, "tumour": 1}

# ---------- Dataset filters which should be chosen below ----------

# --- Renal ccRCC label assignment ---
def _assignCCRCCNormalVsTumour(row):
    if row['tissue'] in ("Normal kidney"):
        return LABELS["normal"]
    elif row['scienceIDType'] == "RCC" and row['tissue'] == "Tumour":
        return LABELS["tumour"]
    else:
        return np.nan

# --- Lung LUAD label assignment ---
LUAD_patient_ids = ('P12', 'P5', 'P39', 'P2', 'P35', 'P32', 'P21', 'P13', 'P33', 'P9', 'P20', 'P38', 'P28', 'P24', 'P16', 'P8', 'P29', 'P34')
def _assignLungLUADNormalVsTumour(row):
    if row['Patient'] in LUAD_patient_ids:
        if row['Celltype (major-lineage)'] == ("Malignant"):
            return LABELS["tumour"]
        else:
            return LABELS['normal']
    else:
        return np.nan

# --- Breast ER+ label assignment ---
def _assignBreastERNormalVsTumour(row):
    if row['subtype'] == 'ER+':
        if row['celltype_major'] == "Cancer Epithelial":
            return LABELS["tumour"]
        elif row['celltype_major'] != 'CAFs':
            return LABELS["normal"]
        else:
            return np.nan
    else:
            return np.nan
    

In [ ]:
# This script processes and trains LightGBM on each train dataset in the same way
# To use a train dataset, change the train path and assign 'dataset_cell_filter' to the relevant dataset filter below

import pickle
from pathlib import Path
import torch
import numpy as np
import scanpy as sc
import scipy.sparse as sp # Used for type checking adata.X
from sklearn.model_selection import train_test_split
from helical.models.scgpt import scGPTConfig, scGPTFineTuningModel
from captum.attr import IntegratedGradients
from collections import defaultdict
from tqdm.auto import tqdm
import pandas as pd
import anndata as ad
# --- Configuration Section ---
print("--- Starting Configuration ---")

# --- Path & filter assignment ---
TRAIN_PATH = Path(r"path_to\lung_train_dataset.h5ad")
dataset_cell_filter = _assignLungLUADNormalVsTumour

# Directories to save gene attributions and trained LGBM model
SHAP_SAVE_DIR = Path(r"path_to\SHAP_save_dir")
MODEL_SAVE_DIR = Path(r"path_to\model_save_dir")


c:\Users\james\.conda\envs\helical-package\Lib\site-packages\louvain\__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
  from .autonotebook import tqdm as notebook_tqdm

INFO:datasets:PyTorch version 2.6.0+cu118 available.


--- Starting Configuration ---


In [ ]:

def prepare_data(path_to_train_dataset: Path, celltype_filter):
    adata = sc.read_h5ad(path_to_train_dataset)
    print(f"[prepare_data] Loaded {adata.n_obs} cells × {adata.n_vars} genes")
        
    # Apply the cell type filter selected above
    adata.obs["label"] = adata.obs.apply(celltype_filter, axis=1)
    adata = adata[adata.obs["label"].notna()].copy()


    downsample_majority = True   
    if downsample_majority:
        # 1. Identify counts
        counts = adata.obs["label"].value_counts()
        maj_label, min_label = counts.idxmax(), counts.idxmin()

        # 2. Split data
        adata_maj = adata[adata.obs["label"] == maj_label].copy()
        adata_rest = adata[adata.obs["label"] != maj_label].copy()

        # 3. Downsample Majority to match Minority size (1:1 ratio)
        target_n = counts[min_label] 
        if len(adata_maj) > target_n:
            sc.pp.subsample(adata_maj, n_obs=target_n, random_state=42)

        # 4. Merge back
        adata = ad.concat([adata_maj, adata_rest], merge="same")

    print(adata.obs['label'].value_counts())
    
    # Ensure integer counts
    if sp.issparse(adata.X):
        adata.X.data = np.rint(adata.X.data).astype(np.int32)
        adata.X = adata.X.astype(np.int32)
    else:
        adata.X = np.rint(adata.X).astype(np.int32)

    print(f"[prepare_data] Before filtering: {adata.n_vars} genes")
    sc.pp.filter_genes(adata, min_counts=1)  # Keep genes expressed at least once
    print(f"[prepare_data] After filtering: {adata.n_vars} genes")
    
    # LGBM-Specific pre-processing
    labels = adata.obs["label"].astype(int).tolist()
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=10000,
        flavor='seurat_v3',  # Works on raw counts
        batch_key=None,
        subset=False  # Just mark HVGs, don't remove genes yet
    )
    print(f"Identified {adata.var['highly_variable'].sum()} highly variable genes.")

    # --- Step 5: Normalize and Convert to Integers ---
    # Normalize BEFORE subsetting so normalization is consistent
    sc.pp.normalize_total(adata, target_sum=1e4)

    # Convert normalized counts back to integers
    if sp.issparse(adata.X):
        adata.X.data = np.rint(adata.X.data).astype(np.int32)
        adata.X = adata.X.astype(np.int32)
    else:
        adata.X = np.rint(adata.X).astype(np.int32)

    # --- Step 6: Subset to HVGs ---
    # Now subset to only the highly variable genes
    adata = adata[:, adata.var['highly_variable']].copy()
    print(f"Subsetted to {adata.n_vars} highly variable genes.")
    
    adata.var["gene_names"] = adata.var.index  # <- will cause issues if gene ids are not in index

    return adata.copy()


ad_tr = prepare_data(TRAIN_PATH, dataset_cell_filter)
y_test = ad_tr.obs["label"].astype(np.int64)


adata_test = ad_tr.copy()
print(f"Test set created with {adata_test.n_obs} cells, {adata_test.n_vars} genes.")

[prepare_data] Loaded 82267 cells × 26077 genes
label
0.0    6087
1.0    6087
Name: count, dtype: int64
[prepare_data] Before filtering: 26077 genes
[prepare_data] After filtering: 23795 genes
Identified 10000 highly variable genes.
Subsetted to 10000 highly variable genes.
Test set created with 12174 cells, 10000 genes.


In [ ]:
from collections import Counter
Counter(list(ad_tr.obs["label"]))

Counter({0.0: 15646, 1.0: 15646})

In [ ]:
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import numpy as np
import pickle
import scipy.sparse as sp
from pathlib import Path
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
import scipy.sparse
import scipy

def preprocess_raw_counts(X_train, X_val, genes):
    """
    Adapted preprocessing specifically for Raw Gene Expression Counts.
    Handles Sparse Matrices and Log-Normalization.
    """
    print(f"\n{'='*60}")
    print(f"Baseline Preprocessing (Raw Counts)")
    print(f"{'='*60}")
    
    # --- STEP 0: Log-Normalization (CRITICAL FOR RAW COUNTS) ---
    print(f"\n0. Log-Normalization (Target Sum: 10,000)")
    
    # 1. Library Size Normalization
    # FIX: Handle sparse matrix sum by removing keepdims and reshaping manually
    counts_per_cell_train = np.array(X_train.sum(axis=1)).reshape(-1, 1)
    counts_per_cell_val = np.array(X_val.sum(axis=1)).reshape(-1, 1)
    
    # Avoid division by zero
    counts_per_cell_train[counts_per_cell_train == 0] = 1
    counts_per_cell_val[counts_per_cell_val == 0] = 1

    # Normalize to 10,000 counts per cell
    scale_factor = 1e4
    
    # Note: If X is sparse, this division might densify it depending on implementation.
    # To remain memory efficient, we can convert to dense here if RAM allows, 
    # as StandardScaler (Step 3) will require dense data anyway.
    if scipy.sparse.issparse(X_train):
        print("   (Converting sparse matrix to dense for scaling)")
        X_train = X_train.toarray()
        X_val = X_val.toarray()

    X_train_norm = (X_train / counts_per_cell_train) * scale_factor
    X_val_norm = (X_val / counts_per_cell_val) * scale_factor
    
    # 2. Log1p Transformation: log(x + 1)
    X_train_norm = np.log1p(X_train_norm)
    X_val_norm = np.log1p(X_val_norm)
    
    print("   Done. Data is now in log1p(CPM) space.")

    # Step 1: Remove genes expressed in <10 cells
    min_cells = 10
    expression_count = (X_train > 0).sum(axis=0)
    keep_mask = expression_count >= min_cells
    
    print(f"\n1. Remove ultra-rare genes (expressed in <{min_cells} cells)")
    print(f"   Keeping {keep_mask.sum()}/{len(genes)} genes")
    
    X_train_filt = X_train_norm[:, keep_mask]
    X_val_filt = X_val_norm[:, keep_mask]
    genes_filt = [g for g, keep in zip(genes, keep_mask) if keep]
    
    # Step 2: Remove low-variance genes (bottom 5%)
    variances = X_train_filt.var(axis=0)
    var_threshold = np.percentile(variances, 5)
    keep_mask2 = variances >= var_threshold
    
    print(f"\n2. Remove low-variance genes (bottom 5%, var < {var_threshold:.6f})")
    print(f"   Keeping {keep_mask2.sum()}/{len(genes_filt)} genes")
    
    X_train_filt = X_train_filt[:, keep_mask2]
    X_val_filt = X_val_filt[:, keep_mask2]
    genes_filt = [g for g, keep in zip(genes_filt, keep_mask2) if keep]
    
    # Step 3: StandardScaler
    print(f"\n3. StandardScaler")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_filt)
    X_val_scaled = scaler.transform(X_val_filt)
    
    # Step 4: Clip extreme outliers
    clip_val = 10
    n_clipped_train = (np.abs(X_train_scaled) > clip_val).sum()
    
    if n_clipped_train > 0:
        print(f"\n4. Clipping {n_clipped_train} extreme values at ±{clip_val}")
        X_train_scaled = np.clip(X_train_scaled, -clip_val, clip_val)
        X_val_scaled = np.clip(X_val_scaled, -clip_val, clip_val)
    
    print(f"\n✓ Final preprocessed Raw Baseline:")
    print(f"  Shape: {X_train_scaled.shape}")
    print(f"  Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.4f}")
    
    return X_train_scaled, X_val_scaled, genes_filt, scaler

X = adata_test.X
y = y_test
if sp.issparse(X):
            X = X.astype(np.float32)
common_genes = adata_test.var['gene_names'].tolist()

# --- 2. Create Train / Validation Split ---
# CRITICAL: Use stratify=y to maintain the Tumor/Non-Tumor ratio in both sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.1,      # 10% of data for validation
    stratify=y,         # Keeps class balance consistent
     random_state=42
)

X_train_final, X_val_final, genes_final, scaler_final = preprocess_raw_counts(X_train, X_val, common_genes)


print(f"Training shape: {X_train_final.shape}")
print(f"Validation shape: {X_val_final.shape}")

# --- 3. Initialize and Train LightGBM ---
lgbm_model = LGBMClassifier(
    objective='binary',
    
    # Reduce model complexity
    max_depth=5,              # Limit tree depth
    num_leaves=31,            # Keep reasonable
    min_child_samples=20,     # More samples per leaf
    
    # Add regularization
    reg_alpha=0.1,            # L1 regularization
    reg_lambda=0.1,           # L2 regularization
    
    # Slow down learning
    learning_rate=0.05,       # Lower learning rate
    n_estimators=500,         # More iterations with early stopping
    
    # Add randomness
    subsample=0.8,            # Use 80% of data per iteration
    colsample_bytree=0.8,     # Use 80% of features per tree
    subsample_freq=5,         # Apply subsampling every 5 iterations
    
    # Other settings
    n_jobs=-1,
    random_state=42,
    verbose=-1
)


print("Training model...")
lgbm_model.fit(
    X_train_final, y_train,
    eval_set=[(X_val_final, y_val)],
    eval_metric='logloss',
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)
y_pred = lgbm_model.predict(X_val_final)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Calculate Accuracy
val_accuracy = accuracy_score(y_val, y_pred)
print(f"\n>>> Validation Accuracy: {val_accuracy:.4f}")

# Detailed Report (Precision, Recall, F1)
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=["Non-Tumour", "Tumour"]))

# --- 5. Gene Attribution ---
# We can directly map importance back to gene names.

feature_imp = pd.DataFrame({
    'Gene': genes_final,
    'Importance': lgbm_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Predictive Genes (based on Embedding Signal Intensity):")
print(feature_imp.head(15))


Baseline Preprocessing (Raw Counts): Geneformer

0. Log-Normalization (Target Sum: 10,000)
   (Converting sparse matrix to dense for scaling)
   Done. Data is now in log1p(CPM) space.

1. Remove ultra-rare genes (expressed in <10 cells)
   Keeping 8363/10000 genes

2. Remove low-variance genes (bottom 5%, var < 0.007860)
   Keeping 7944/8363 genes

3. StandardScaler

4. Clipping 119994 extreme values at ±10

✓ Final preprocessed Raw Baseline:
  Shape: (10956, 7944)
  Mean: -0.006755, Std: 0.8946
Training shape: (10956, 7944)
Validation shape: (1218, 7944)
Training model...
Training until validation scores don't improve for 50 rounds


  warnings.warn(



Early stopping, best iteration is:
[250]	valid_0's binary_logloss: 0.0632259

>>> Validation Accuracy: 0.9770

Classification Report:
              precision    recall  f1-score   support

  Non-Tumour       0.98      0.98      0.98       609
      Tumour       0.98      0.98      0.98       609

    accuracy                           0.98      1218
   macro avg       0.98      0.98      0.98      1218
weighted avg       0.98      0.98      0.98      1218


Top 15 Predictive Genes (based on Embedding Signal Intensity):
         Gene  Importance
6442    SFTPB         126
7349    MUC5B          66
4581  IL13RA2          64
3662     KRT7          56
6573  SCGB3A2          50
1088    S100P          49
3427  SLC34A2          47
104      CD74          47
1765  CEACAM5          41
412     KRT19          39
200    TYROBP          32
7400    C4BPA          31
2861   ATP1B1          31
7346   SFTPA1          29
247      SLPI          27


In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

shap_save_dir = SHAP_SAVE_DIR
shap_save_dir.mkdir(parents=True, exist_ok=True)

explainer = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_train_final)

train_genes = genes_final
# Check format
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Use tumor class

print(f"SHAP values shape: {shap_values.shape}")  # Should be (9052, 6939)

# Calculate mean absolute SHAP value per gene
mean_shap = np.abs(shap_values).mean(axis=0)  # Shape: (6939,)

print(f"Mean SHAP shape: {mean_shap.shape}")
print(f"Common genes length: {len(train_genes)}")

# Create importance dataframe
shap_importance = pd.DataFrame({
    'Gene': train_genes,
    'SHAP_Importance': mean_shap
}).sort_values('SHAP_Importance', ascending=False)

print("\nTop 15 Genes by SHAP Importance:")
print(shap_importance.head(15))

# Summary plot (beeswarm) - shows distribution of impacts
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_train_final, feature_names=train_genes,
                  max_display=20, show=False)
plt.tight_layout()
plt.savefig(shap_save_dir / 'shap_summary_plot.png', dpi=300, bbox_inches='tight')
plt.close()

# Bar plot - shows average importance
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_train_final, feature_names=train_genes,
                  plot_type="bar", max_display=20, show=False)
plt.tight_layout()
plt.savefig(shap_save_dir / 'shap_bar_plot.png', dpi=300, bbox_inches='tight')
plt.close()

# Dependence plot for top gene
top_gene = shap_importance.iloc[0]['Gene']
top_gene_idx = train_genes.index(top_gene)

plt.figure(figsize=(10, 6))
shap.dependence_plot(top_gene_idx, shap_values, X_train_final,
                     feature_names=train_genes, show=False)
plt.title(f'SHAP Dependence Plot: {top_gene}')
plt.tight_layout()
plt.savefig(shap_save_dir / f'shap_dependence_{top_gene}.png', dpi=300, bbox_inches='tight')
plt.close()

# Compare SHAP importance vs LightGBM feature importance
comparison = pd.DataFrame({
    'Gene': train_genes,
    'LightGBM_Importance': lgbm_model.feature_importances_,
    'SHAP_Importance': mean_shap
}).sort_values('SHAP_Importance', ascending=False)

print(f"\nCorrelation between LightGBM and SHAP importance: {comparison['LightGBM_Importance'].corr(comparison['SHAP_Importance']):.3f}")

comparison.to_csv(shap_save_dir / 'importance_comparison.csv', index=False)

# Save SHAP values for later use
np.save(shap_save_dir / 'shap_values.npy', shap_values)
shap_importance.to_csv(shap_save_dir / 'shap_importance.csv', index=False)

print(f"\nSHAP analysis complete! Results saved to {shap_save_dir}")

  warnings.warn(

  shap.summary_plot(shap_values, X_train_final, feature_names=train_genes,



SHAP values shape: (10956, 7944)
Mean SHAP shape: (7944,)
Common genes length: 7944

Top 15 Genes by SHAP Importance:
         Gene  SHAP_Importance
1088    S100P         0.560170
6442    SFTPB         0.559916
3662     KRT7         0.526146
7349    MUC5B         0.477349
4581  IL13RA2         0.412704
200    TYROBP         0.262954
412     KRT19         0.243146
2861   ATP1B1         0.197608
1765  CEACAM5         0.195451
2534     AQP1         0.180094
237    LAPTM5         0.176016
6573  SCGB3A2         0.161188
104      CD74         0.157784
3427  SLC34A2         0.144264
92     FCER1G         0.124614


  shap.summary_plot(shap_values, X_train_final, feature_names=train_genes,




Correlation between LightGBM and SHAP importance: 0.857

SHAP analysis complete! Results saved to C:\Users\james\scRNA\scRNArena\classification\LGBM\SHAP_LUADTrain


<Figure size 1000x600 with 0 Axes>

In [ ]:
save_dir = MODEL_SAVE_DIR
from pathlib import Path
import pickle
import json

save_dir.mkdir(parents=True, exist_ok=True)

# Save the trained model
with open(save_dir / 'lgbm_model.pkl', 'wb') as f:
    pickle.dump(lgbm_model, f)

metadata = {
    'common_genes': genes_final,
    'n_features': len(genes_final),
    'val_accuracy': float(val_accuracy),
    'model_params': lgbm_model.get_params()
}

with open(save_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)